<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Introduction to LangChain on Databricks
#### Duration: 20 minutes
## Overview
In this demo, you will build a single Data Analyst agent using LangChain on Databricks. You will define a custom tool with LangChain's `@tool` decorator, register it to Unity Catalog, discover it via a Managed MCP server, and connect it to a LangChain agent with MLflow tracing.

## Learning Objectives
By the end of this demo, you will be able to:
1. Define custom tools using LangChain's `@tool` decorator
2. Build and test a LangChain agent with `create_agent`
3. Register tools to Unity Catalog for governance
4. Discover UC tools via Managed MCP servers
5. Rebuild the agent with MCP-sourced tools
6. Enable MLflow autologging and inspect execution traces


<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
This notebook mirrors notebook <strong>05 Demo - Introduction to OpenAI Agents SDK on Databricks</strong>, using LangChain instead of the OpenAI Agents SDK.
</p>
</div>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 6</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V6</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup

In [0]:
%run ./Includes/Classroom-Setup-7

Installing 14 package(s) with uv...
  pyyaml==6.0.3
  langgraph==1.2.12
  langgraph-prebuilt==1.1.0
  langchain==1.4.3
  langchain-core==1.6.6
  databricks-langchain==0.20.0
  databricks-sdk==0.145.0
  nest_asyncio==1.6.0
  databricks-mcp==0.9.2
  unitycatalog-ai[databricks]==0.3.2
  mlflow==3.16.1
  mlflow-skinny==3.16.1
  mlflow-tracing==3.16.1
  mcp==1.30.0
Library installation complete.


Starting Demo Environment Setup
Using config: Includes/config/config-7.yaml

[0] Compute check passed: Serverless 6 detected.

[1] Setting up catalog...
Vocareum workspace detected.
  Catalog 'labuser17579966_1791942688' found. Using it.

[2] Setting up schema...
  Catalog : labuser17579966_1791942688
  Schema  : agent_training

[3] Setting up Unity Gateway model service...
  Model service created: labuser17579966_1791942688.agent_training.agent-training-model → databricks-gpt-5-2

[4] Processing datasets...
  Using Vocareum Delta Share: airbnb

[5] Setting up MLflow...
  Setting tracking URI: databricks
Setting up experiment: langchain-agent-mcp-training
  Created experiment: /Workspace/Users/labuser17579966_1791942688@vocareum.com/langchain-agent-mcp-training
  Active experiment: /Workspace/Users/labuser17579966_1791942688@vocareum.com/langchain-agent-mcp-training
Refreshing environment

Demo Environment Setup Complete


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


Catalog,"labuser17579966_1791942688<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791942688';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"
Schema,agent_trainingCopy
Table,"labuser17579966_1791942688.agent_training.sf_airbnb_listings<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791942688.agent_training.sf_airbnb_listings';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"
Model service,"labuser17579966_1791942688.agent_training.agent-training-model<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791942688.agent_training.agent-training-model';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"
MLflow experiment,"/Workspace/Users/labuser17579966_1791942688@vocareum.com/langchain-agent-mcp-training<button onclick=""var t=document.createElement('textarea');t.value='/Workspace/Users/labuser17579966_1791942688@vocareum.com/langchain-agent-mcp-training';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"


### A1. Import Libraries

We import the core components needed for this demo:
- `ChatDatabricks` for connecting to Databricks model serving endpoints
- `@tool` decorator for defining agent tools
- `create_agent` from `langchain.agents` for building the agent

In [0]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from databricks_langchain import ChatDatabricks

print("LangChain components imported.")

LangChain components imported.


## B. Define the Data Analyst Agent

### B1. Define the `classify_price_tier` Tool

The `classify_price_tier` tool classifies an Airbnb listing's nightly price into one of four market tiers based on the SF pricing distribution:

| Tier | Price Range | Percentile Band |
|---|---|---|
| **Budget** | Under $100 | Bottom 25% |
| **Mid-Range** | $100 -- $174 | 25--50% |
| **Premium** | $175 -- $299 | 50--75% |
| **Luxury** | $300+ | Top 25% |

LangChain's `@tool` decorator registers this function as a tool that the agent can invoke. The SDK extracts the function signature and docstring to generate the tool schema that the LLM uses to decide when and how to call the tool.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
An example production workflow is: prototype with <code>@tool</code>, then register to Unity Catalog, then access via a Managed MCP server. This demo follows that full progression. In Section C, we register the function to Unity Catalog. In Section D, we connect via MCP.
</p>
</div>
</div>
</div>

In [0]:
import json

@tool
def classify_price_tier(price: float) -> str:
    """Classify an Airbnb listing price into a market tier based on SF pricing distribution.

    Uses hardcoded SF Airbnb market thresholds to label a nightly price as
    Budget, Mid-Range, Premium, or Luxury, and returns the corresponding
    percentile band for context.

    Useful for questions like: 'Is $180/night a good deal in SF?',
    'What tier does this listing fall into?'

    Returns a JSON object with tier, percentile_band, and a short interpretation.

    Args:
        price: Nightly listing price in USD.
    """
    tiers = [
        (100,  "Budget",    "bottom 25%",  "Well below the SF market average."),
        (175,  "Mid-Range", "25-50%",      "Around the SF market median."),
        (300,  "Premium",   "50-75%",      "Above average for SF listings."),
        (float("inf"), "Luxury", "top 25%", "Among the most expensive listings in SF."),
    ]

    if price < 0:
        return json.dumps({"error": "Price must be a positive value."})

    for threshold, tier, band, interpretation in tiers:
        if price < threshold:
            return json.dumps({
                "price": price,
                "tier": tier,
                "percentile_band": band,
                "interpretation": interpretation
            }, indent=2)

print(f"Defined tool: {classify_price_tier.name}")

Defined tool: classify_price_tier


### B2. Create the Agent

With the tool defined, we create an agent using LangChain's `create_agent`. This takes three inputs:
1. **LLM**: `ChatDatabricks` connects to a Databricks model serving endpoint
2. **Tools**: The list of tools the agent can call
3. **System prompt**: Instructions that shape the agent's behavior, passed as `system_prompt`

The agent runs a ReAct loop: the LLM reasons about which tool to call, executes it, observes the result, and generates a final response. `create_agent` replaces LangGraph's `create_react_agent`, which is deprecated as of LangGraph 1.0, and still runs on LangGraph under the hood.

In [0]:
import yaml
from pathlib import Path

agent_files = {
    "da": "data_analyst_agent"
}

agents = {}
for prefix, filename in agent_files.items():
    cfg = yaml.safe_load(Path(f"./agent_config/{filename}.yaml").read_text())
    agents[prefix] = {
        "name": cfg["name"],
        "llm_endpoint": cfg["llm_endpoint"],
        "system_prompt": cfg["system_prompt"],
    }

In [0]:
system_prompt = agents["da"]["system_prompt"]
llm_endpoint = f"{catalog_name}.{schema_name}.{agents["da"]["llm_endpoint"]}"

llm = ChatDatabricks(
    model=llm_endpoint,
    use_ai_gateway=True
)

agent_executor = create_agent(llm, tools=[classify_price_tier], system_prompt=system_prompt)

print("Agent created with inline @tool.")

Agent created with inline @tool.


### B3. Test the Agent

We send three progressively complex queries to validate that the agent correctly invokes the tool and interprets the results.

1. A straightforward pricing question that requires a single tool call.

In [0]:
query_1 = "Is $150/night a good deal in SF?"
result_1 = agent_executor.invoke({"messages": [("human", query_1)]})
print(f"User:     {query_1}")
print(f"Response: {result_1['messages'][-1].content}")

User:     Is $150/night a good deal in SF?
Response: $150/night in San Francisco is **Mid-Range**, sitting in roughly the **25th–50th percentile** of SF Airbnb prices—**around the market median**.

**Bottom line:** It’s a **fair/typical price**, not a standout bargain. It becomes a “good deal” if the place has strong location, amenities, or low extra fees.


2. A direct tier classification at the high end of the price spectrum.

In [0]:
query_2 = "What tier would a $500/night luxury listing fall into?"
result_2 = agent_executor.invoke({"messages": [("human", query_2)]})
print(f"User:     {query_2}")
print(f"Response: {result_2['messages'][-1].content}")

User:     What tier would a $500/night luxury listing fall into?
Response: $500/night in San Francisco falls into the **Luxury** tier.

- **Price:** $500/night  
- **Tier:** **Luxury**  
- **Market context:** **Top 25%** of SF listings (among the most expensive)


3. Multiple prices in a single request, testing whether the agent calls the tool multiple times.

In [0]:
query_3 = "Classify these prices: $80, $200, $350"
result_3 = agent_executor.invoke({"messages": [("human", query_3)]})
print(f"User:     {query_3}")
print(f"Response: {result_3['messages'][-1].content}")

User:     Classify these prices: $80, $200, $350
Response: - **$80/night:** **Budget** *(bottom 25%)* — well below the SF market average.  
- **$200/night:** **Premium** *(50–75%)* — above average for SF listings.  
- **$350/night:** **Luxury** *(top 25%)* — among the most expensive SF listings.


## C. Register Tools to Unity Catalog

### C1. Register `classify_price_tier` as a Python UC Function

The inline `@tool` is local to this notebook session. To make the tool governed, discoverable, and reusable across agents and workspaces, we register it to Unity Catalog as a Python function using `DatabricksFunctionClient`.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
The function registered below is a <strong>plain Python function</strong> (no <code>@tool</code> decorator). The <code>DatabricksFunctionClient</code> inspects the function signature and docstring to create the UC function metadata. The logic is identical to the tool we prototyped above.
</p>
</div>
</div>
</div>

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

def classify_price_tier_uc(price: float) -> str:
    """Classify an Airbnb listing price into a market tier based on SF pricing distribution.

    Uses hardcoded SF Airbnb market thresholds to label a nightly price as
    Budget, Mid-Range, Premium, or Luxury, and returns the corresponding
    percentile band for context.

    Useful for questions like: 'Is $180/night a good deal in SF?',
    'What tier does this listing fall into?'

    Returns a JSON object with tier, percentile_band, and a short interpretation.

    Args:
        price: Nightly listing price in USD.
    """
    import json

    tiers = [
        (100,  "Budget",    "bottom 25%",  "Well below the SF market average."),
        (175,  "Mid-Range", "25-50%",      "Around the SF market median."),
        (300,  "Premium",   "50-75%",      "Above average for SF listings."),
        (float("inf"), "Luxury", "top 25%", "Among the most expensive listings in SF."),
    ]

    if price < 0:
        return json.dumps({"error": "Price must be a positive value."})

    for threshold, tier, band, interpretation in tiers:
        if price < threshold:
            return json.dumps({
                "price": price,
                "tier": tier,
                "percentile_band": band,
                "interpretation": interpretation
            }, indent=2)

client = DatabricksFunctionClient()
function_info = client.create_python_function(
    func=classify_price_tier_uc,
    catalog=catalog_name,
    schema=schema_name,
    replace=True,
)

print(f"Created UC function: {function_info.full_name}")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-8cbb9a5d-e6dc-41ac-9899-4ec6ed528166/lib/python3.12/site-packages/databricks/connect/session.py:476: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


Created UC function: labuser17579966_1791942688.agent_training.classify_price_tier_uc


### C2. Verify Registered Functions

Confirm that the function is visible in the Unity Catalog schema, then call it directly to verify it returns the expected output.
1. Navigate to the **Catalog Explorer** on the left side menu and search for `classify_price_tier_uc`
1. Run the next cell to test the output from calling the function from UC

In [0]:
result = spark.sql(f"SELECT {catalog_name}.{schema_name}.classify_price_tier_uc(150.0) AS result")
display(result)

result
"{ ""price"": 150.0, ""tier"": ""Mid-Range"", ""percentile_band"": ""25-50%"", ""interpretation"": ""Around the SF market median."" }"


## D. Discover Tools via MCP

### D1. Connect to the Managed MCP Server

Databricks provides **Managed MCP servers** that automatically expose Unity Catalog functions as tools. Any agent can discover and call these functions through the MCP protocol without custom integration code.

The MCP server endpoint for UC functions follows this pattern:
`https://workspace/api/2.0/mcp/functions/catalog/schema`

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
We use <code>DatabricksMCPClient</code> from <code>databricks_mcp</code> to interact with the MCP endpoint in a notebook context.
See: Managed MCP servers (<a href="https://docs.databricks.com/aws/en/generative-ai/mcp/managed-mcp" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/generative-ai/mcp/managed-mcp" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/generative-ai/mcp/managed-mcp" style="color: #1976d2; text-decoration: underline;">GCP</a>)
</p>
</div>
</div>
</div>

In [0]:
import nest_asyncio
nest_asyncio.apply()

from databricks_mcp import DatabricksMCPClient
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()
workspace_host = ws.config.host

# Point to the MCP server for our catalog/schema
mcp_url = f"{workspace_host}/api/2.0/mcp/functions/{catalog_name}/{schema_name}"
mcp_client = DatabricksMCPClient(server_url=mcp_url, workspace_client=ws)

# Discover available tools
tools_discovered = mcp_client.list_tools()
print(f"Discovered {len(tools_discovered)} UC function tool(s) via MCP:")
for t in tools_discovered:
    print(f"  - {t.name}: {t.description}")

Discovered 1 UC function tool(s) via MCP:
  - labuser17579966_1791942688__agent_training__classify_price_tier_uc: Classify an Airbnb listing price into a market tier based on SF pricing distribution. Uses hardcoded SF Airbnb market thresholds to label a nightly price as Budget, Mid-Range, Premium, or Luxury, and returns the corresponding percentile band for context. Useful for questions like: "Is $180/night a good deal in SF?", "What tier does this listing fall into?" Returns a JSON object with tier, percentile_band, and a short interpretation.


### D2. Call the UC Function Through MCP

Now we call the registered UC function through the MCP protocol, just as an agent would. The default tool name is of the form `catalogName__schemaName__toolName`.

In [0]:
tool_name = f"{catalog_name}__{schema_name}__classify_price_tier_uc"
result = mcp_client.call_tool(tool_name, {"price": 120.00})
print(f"Result: {''.join([c.text for c in result.content])}")

Result: {"is_truncated":false,"columns":["output"],"rows":[["{\n  \"price\": 120.0,\n  \"tier\": \"Mid-Range\",\n  \"percentile_band\": \"25-50%\",\n  \"interpretation\": \"Around the SF market median.\"\n}"]]}


### D3. Load MCP Tools into LangChain

We convert the MCP tools we discovered into LangChain-compatible `StructuredTool` objects. Each tool wraps a call to `mcp_client.call_tool()`, so tool invocations route through MCP at runtime.

<div style="display:flex;align-items:center;gap:0;margin:20px 0;flex-wrap:wrap;font-family:system-ui,sans-serif;">
  <div style="background:#F9F7F4;border:2px solid #1B5162;border-radius:8px;padding:10px 14px;text-align:center;font-size:14px;font-weight:600;">@tool<br/><span style="font-weight:400;font-size:12px;">Local only</span></div>
  <div style="color:#999;font-size:22px;padding:0 8px;">&#x2192;</div>
  <div style="background:#2574B5;color:#fff;border-radius:8px;padding:10px 14px;text-align:center;font-size:14px;font-weight:600;">UC Function<br/><span style="font-weight:400;font-size:12px;">Governed + reusable</span></div>
  <div style="color:#999;font-size:22px;padding:0 8px;">&#x2192;</div>
  <div style="background:#02A36F;color:#fff;border-radius:8px;padding:10px 14px;text-align:center;font-size:14px;font-weight:600;">MCP Server<br/><span style="font-weight:400;font-size:12px;">Agent-accessible + standardized</span></div>
  <div style="color:#999;font-size:22px;padding:0 8px;">&#x2192;</div>
  <div style="background:#FF5F46;color:#fff;border-radius:8px;padding:10px 14px;text-align:center;font-size:14px;font-weight:600;">LangChain Agent<br/><span style="font-weight:400;font-size:12px;">Production-ready</span></div>
</div>

This is the recommended progression: start with `@tool` for rapid prototyping, register to UC for governance, then expose through MCP for production agent access.

The next cell defines `_make_mcp_tool`, which converts each MCP tool into a LangChain `StructuredTool`. Before you run it, click each pill at the top to highlight that part of the code and read what it does.

<div id="mcp-tool-explorer"></div>
<style>
  .sx-wrap-py {
    position: relative;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
    border: 1px solid #DCE0E2;
    border-radius: 12px;
    overflow: hidden;
    background: #F9F7F4;
    margin: 16px auto;
    max-width: 100%;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
  }
  .sx-wrap-py .sx-title {
    display: flex;
    justify-content: space-between;
    align-items: center;
    background: #EEEDE9;
    border-bottom: 1px solid #DCE0E2;
    padding: 10px 16px;
    font-size: 14px;
    font-weight: 700;
    color: #5A6F77;
    letter-spacing: 0.02em;
  }
  .sx-wrap-py .sx-rail {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    padding: 12px 16px;
    background: #F9F7F4;
    border-bottom: 1px solid #EEEDE9;
  }
  .sx-wrap-py .sx-chip {
    border: 1px solid #DCE0E2;
    background: #FFFFFF;
    border-radius: 999px;
    padding: 6px 12px;
    font-size: 14px;
    font-weight: 700;
    color: #0b2026;
    cursor: pointer;
    transition: background 0.12s, border-color 0.12s, color 0.12s;
  }
  .sx-wrap-py .sx-chip:hover,
  .sx-wrap-py .sx-chip.active {
    background: #E4F0FA;
    border-color: #2272B4;
    color: #1B5162;
  }
  .sx-wrap-py .sx-code {
    padding: 12px 0;
    overflow-x: auto;
    font-size: 14px;
    line-height: 1.6;
    background: #F9F7F4;
  }
  .sx-wrap-py .sx-code,
  .sx-wrap-py .sx-line-code {
    font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace !important;
  }
  .sx-wrap-py .sx-line {
    display: grid;
    grid-template-columns: 50px 1fr;
  }
  .sx-wrap-py .sx-line-no {
    color: #618794;
    text-align: right;
    padding: 0 12px;
    user-select: none;
    border-right: 1px solid #EEEDE9;
    font-weight: 700;
    font-size: 14px;
  }
  .sx-wrap-py .sx-line-code {
    padding: 0 16px;
    white-space: pre;
    color: #0b2026;
  }
  .sx-wrap-py .sx-hotspot { cursor: pointer; }
  .sx-wrap-py .sx-line.sx-active-line { background: #E4F0FA; }
  .sx-wrap-py .sx-active-line .sx-line-code {
    border-left: 3px solid #2272B4;
    padding-left: 13px;
  }
  .sx-wrap-py .sx-active-line .sx-line-no { color: #1B5162; }
  .sx-wrap-py .sx-overlay {
    display: none;
    position: absolute;
    right: 18px;
    top: 110px;
    width: 340px;
    max-width: calc(100% - 36px);
    background: #E4F0FA;
    border: 1px solid #4299E0;
    border-left: 5px solid #2272B4;
    border-radius: 10px;
    box-shadow: 0 6px 18px rgba(27,49,57,0.18), 0 16px 36px rgba(27,49,57,0.14);
    padding: 14px 16px;
    z-index: 5;
  }
  .sx-wrap-py .sx-overlay-title {
    font-weight: 800;
    margin-bottom: 8px;
    color: #1B5162;
    font-size: 16px;
  }
  .sx-wrap-py .sx-overlay-body {
    color: #0b2026;
    line-height: 1.5;
    font-size: 14px;
  }
  .sx-wrap-py .sx-overlay-body code {
    font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace;
    background: #FFFFFF;
    border: 1px solid #C4CCD6;
    color: #1B5162;
    padding: 1px 5px;
    border-radius: 4px;
    font-size: 14px;
  }
</style>
<script>
(function () {
  var config = {
  "mountId": "mcp-tool-explorer",
  "title": "PYTHON",
  "hotspots": [
    {
      "title": "imports",
      "startLine": 1,
      "endLine": 2,
      "description": "<code>StructuredTool</code> is LangChain's tool class for a function with a typed argument schema. Pydantic's <code>create_model</code> and <code>Field</code> build that schema at runtime, because we only learn the arguments when MCP tells us."
    },
    {
      "title": "type mapping",
      "startLine": 4,
      "endLine": 5,
      "description": "MCP servers describe each argument with a JSON Schema type such as <code>\"number\"</code> or <code>\"string\"</code>. This lookup turns those into the Python types LangChain expects. Anything unrecognized falls back to <code>str</code>."
    },
    {
      "title": "one tool at a time",
      "startLine": 7,
      "endLine": 9,
      "description": "The function converts a single tool returned by <code>list_tools()</code>. Its name has the form <code>catalog__schema__function</code>, and it becomes the tool name the LLM sees."
    },
    {
      "title": "read the input schema",
      "startLine": 11,
      "endLine": 12,
      "description": "<code>inputSchema</code> is the JSON Schema the MCP server returned for this tool. Databricks generates it from the Unity Catalog function's parameters."
    },
    {
      "title": "build each field",
      "startLine": 13,
      "endLine": 19,
      "description": "Each argument becomes a <code>(python_type, Field(description=...))</code> pair. The description comes from the UC function's documentation, so the docstring you wrote when registering it reaches the LLM here."
    },
    {
      "title": "args model",
      "startLine": 20,
      "endLine": 20,
      "description": "<code>create_model</code> builds a Pydantic class on the fly, for example <code>classify_price_tier_uc_args(price: float)</code>. LangChain turns it into the argument schema the LLM fills in, and Pydantic validates the values before the call."
    },
    {
      "title": "forward calls to MCP",
      "startLine": 22,
      "endLine": 25,
      "description": "The LangChain tool has no logic of its own. <code>_invoke</code> forwards the arguments to <code>mcp_client.call_tool()</code> and joins the text parts of the MCP response. The UC function still runs in Databricks, under Unity Catalog permissions."
    },
    {
      "title": "assemble the tool",
      "startLine": 27,
      "endLine": 32,
      "description": "<code>StructuredTool.from_function</code> combines the pieces: the function to run, the tool name, the description from the UC function's comment, and the argument schema."
    },
    {
      "title": "convert every tool",
      "startLine": 34,
      "endLine": 38,
      "description": "Apply the wrapper to every tool the MCP server exposed, then list the results. These tools are passed to <code>create_agent</code> in section E, exactly like the inline <code>@tool</code> earlier."
    }
  ],
  "source": "from langchain_core.tools import StructuredTool\nfrom pydantic import Field, create_model\n\n# MCP describes arguments with JSON Schema types; LangChain needs Python types\nJSON_TO_PYTHON = {\"string\": str, \"number\": float, \"integer\": int, \"boolean\": bool}\n\ndef _make_mcp_tool(mcp_tool_meta):\n    \"\"\"Wrap a single MCP tool as a LangChain StructuredTool.\"\"\"\n    tool_name = mcp_tool_meta.name\n\n    # Build a Pydantic model from the MCP tool's input schema\n    schema_props = mcp_tool_meta.inputSchema.get(\"properties\", {})\n    field_definitions = {}\n    for prop_name, prop_info in schema_props.items():\n        prop_type = JSON_TO_PYTHON.get(prop_info.get(\"type\", \"string\"), str)\n        field_definitions[prop_name] = (\n            prop_type,\n            Field(description=prop_info.get(\"description\", \"\")),\n        )\n    args_model = create_model(f\"{tool_name}_args\", **field_definitions)\n\n    # Every call to the LangChain tool is forwarded to the MCP server\n    def _invoke(**kwargs):\n        result = mcp_client.call_tool(tool_name, kwargs)\n        return \"\".join([c.text for c in result.content])\n\n    return StructuredTool.from_function(\n        func=_invoke,\n        name=tool_name,\n        description=mcp_tool_meta.description or \"\",\n        args_schema=args_model,\n    )\n\nmcp_langchain_tools = [_make_mcp_tool(t) for t in tools_discovered]\n\nprint(f\"Loaded {len(mcp_langchain_tools)} MCP tool(s) as LangChain tools:\")\nfor t in mcp_langchain_tools:\n    print(f\"  - {t.name}\")"
};
  function escapeHtml(v) {
    return String(v).replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;");
  }
  var mount = document.getElementById(config.mountId);
  var lines = config.source.split("\n");
  var hotspots = config.hotspots;
  function hotspotFor(lineNo) {
    for (var i = 0; i < hotspots.length; i++) {
      if (lineNo >= hotspots[i].startLine && lineNo <= hotspots[i].endLine) return { h: hotspots[i], i: i };
    }
    return null;
  }
  var codeHtml = lines.map(function(line, idx) {
    var lineNo = idx + 1;
    var hs = hotspotFor(lineNo);
    var cls = hs ? "sx-line sx-hotspot sx-hotspot-" + hs.i : "sx-line";
    var attr = hs ? ' data-hotspot-index="' + hs.i + '"' : "";
    return '<div class="' + cls + '"' + attr + '><div class="sx-line-no">' + lineNo + '</div><div class="sx-line-code">' + escapeHtml(line) + '</div></div>';
  }).join("");
  var railHtml = '<div class="sx-rail">' + hotspots.map(function(h, i) {
    return '<button class="sx-chip" data-hotspot-index="' + i + '">' + escapeHtml(h.title) + '</button>';
  }).join("") + '</div>';
  mount.innerHTML = '<div class="sx-wrap-py"><div class="sx-title"><span>' + escapeHtml(config.title) + '</span></div>' + railHtml + '<div class="sx-code">' + codeHtml + '</div><div class="sx-overlay"><div class="sx-overlay-title"></div><div class="sx-overlay-body"></div></div></div>';
  var overlay = mount.querySelector(".sx-overlay");
  var overlayTitle = mount.querySelector(".sx-overlay-title");
  var overlayBody = mount.querySelector(".sx-overlay-body");
  var activeIdx = null;
  function clearActive() {
    mount.querySelectorAll(".sx-active-line").forEach(function(el) { el.classList.remove("sx-active-line"); });
    mount.querySelectorAll(".sx-chip.active").forEach(function(el) { el.classList.remove("active"); });
  }
  function activate(idx) {
    clearActive();
    if (idx === null) return;
    mount.querySelectorAll(".sx-hotspot-" + idx).forEach(function(el) { el.classList.add("sx-active-line"); });
    mount.querySelector('.sx-chip[data-hotspot-index="' + idx + '"]').classList.add("active");
  }
  function showHotspot(idx, triggerEl) {
    if (overlay.style.display === "block" && activeIdx === idx) {
      overlay.style.display = "none"; activeIdx = null; clearActive(); return;
    }
    var h = hotspots[idx];
    overlayTitle.textContent = h.title;
    overlayBody.innerHTML = h.description;
    overlay.style.display = "block";
    var wrap = mount.querySelector(".sx-wrap-py");
    var trigger = triggerEl || mount.querySelector(".sx-hotspot-" + idx);
    if (trigger) {
      var wrapRect = wrap.getBoundingClientRect();
      var trigRect = trigger.getBoundingClientRect();
      var top = trigRect.top - wrapRect.top;
      overlay.style.top = Math.max(18, Math.min(top, wrap.clientHeight - overlay.offsetHeight - 18)) + "px";
    }
    activeIdx = idx;
    activate(idx);
  }
  mount.querySelectorAll(".sx-hotspot").forEach(function(el) {
    el.addEventListener("click", function() { showHotspot(Number(el.dataset.hotspotIndex), el); });
  });
  mount.querySelectorAll(".sx-chip").forEach(function(chip) {
    chip.addEventListener("click", function() { showHotspot(Number(chip.dataset.hotspotIndex), null); });
  });
  mount.querySelector(".sx-wrap-py").addEventListener("click", function(e) {
    if (e.target.closest(".sx-hotspot") || e.target.closest(".sx-chip") || e.target.closest(".sx-overlay")) return;
    overlay.style.display = "none"; activeIdx = null; clearActive();
  });
  showHotspot(0, null);
})();
</script>

In [0]:
from langchain_core.tools import StructuredTool
from pydantic import Field, create_model

# MCP describes arguments with JSON Schema types; LangChain needs Python types
JSON_TO_PYTHON = {"string": str, "number": float, "integer": int, "boolean": bool}

def _make_mcp_tool(mcp_tool_meta):
    """Wrap a single MCP tool as a LangChain StructuredTool."""
    # tool_name = mcp_tool_meta.name
    original_name = mcp_tool_meta.name
    tool_name = original_name[:64]  # OpenAI enforces a 64-char limit on tool names

    # Build a Pydantic model from the MCP tool's input schema
    schema_props = mcp_tool_meta.inputSchema.get("properties", {})
    field_definitions = {}
    for prop_name, prop_info in schema_props.items():
        prop_type = JSON_TO_PYTHON.get(prop_info.get("type", "string"), str)
        field_definitions[prop_name] = (
            prop_type,
            Field(description=prop_info.get("description", "")),
        )
    args_model = create_model(f"{tool_name}_args", **field_definitions)

    # Every call to the LangChain tool is forwarded to the MCP server
    def _invoke(**kwargs):
        result = mcp_client.call_tool(original_name, kwargs)  # MCP needs the full name
        return "".join([c.text for c in result.content])

    return StructuredTool.from_function(
        func=_invoke,
        name=tool_name,
        description=mcp_tool_meta.description or "",
        args_schema=args_model,
    )

mcp_langchain_tools = [_make_mcp_tool(t) for t in tools_discovered]

print(f"Loaded {len(mcp_langchain_tools)} MCP tool(s) as LangChain tools:")
for t in mcp_langchain_tools:
    print(f"  - {t.name}")

Loaded 1 MCP tool(s) as LangChain tools:
  - labuser17579966_1791942688__agent_training__classify_price_tier_


## E. Enable MLflow Tracing and Run the MCP-Powered Agent

### E1. Enable Autologging and UC Registration

MLflow autologging for LangChain automatically captures execution traces for every agent run. Each trace records the full sequence of LLM calls, tool invocations, inputs, outputs, and latencies without requiring any code changes to the agent itself.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
When using LangChain, use <code>mlflow.langchain.autolog()</code>. Autologging captures: LLM call spans (model, prompt, completion tokens), tool call spans (function name, arguments, return value), and the overall agent run span with total latency.
</p>
</div>
</div>
</div>

In [0]:
import mlflow
from mlflow.entities.trace_location import UnityCatalog

mlflow.langchain.autolog()

uc = UnityCatalog(
    catalog_name = catalog_name,
    schema_name = schema_name,
    table_prefix = "lc_sa" # oai_sa: "OpenAI Single Agent"
)

experiment_path = f"/Users/{username}/langchain-sdk-training"

mlflow.set_experiment(
    experiment_name=experiment_path,
    trace_location=uc,
)

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/3188534432723517', creation_time=1791949439731, effective_trace_archival_retention=None, experiment_id='3188534432723517', last_update_time=1791949439731, lifecycle_stage='active', name='/Users/labuser17579966_1791942688@vocareum.com/langchain-sdk-training', tags={'mlflow.experiment.databricksTelemetryDestinationId': '60a9271f-2484-303c-84f0-9677daa1d128',
 'mlflow.experiment.databricksTraceAnnotationStorageTable': 'labuser17579966_1791942688.agent_training.lc_sa_otel_annotations',
 'mlflow.experiment.databricksTraceDestinationPath': 'labuser17579966_1791942688.agent_training.lc_sa',
 'mlflow.experiment.databricksTraceLogStorageTable': 'labuser17579966_1791942688.agent_training.lc_sa_otel_logs',
 'mlflow.experiment.databricksTraceSpanStorageTable': 'labuser17579966_1791942688.agent_training.lc_sa_otel_spans',
 'mlflow.experiment.databricksTraceStorageTable': 'labuser17579966_1791942688.agent_training.mlflow_experiment_tra

### E2. Build the MCP-Powered Agent

We rebuild the agent using the same LLM and system prompt, but now the tools come from MCP rather than inline `@tool` definitions. The `create_agent` call is identical. 

In [0]:
mcp_agent_executor = create_agent(
    llm, 
    tools=mcp_langchain_tools, 
    system_prompt=system_prompt
    )

print("Agent created with MCP-sourced tools.")
print(f"  Tools: {[t.name for t in mcp_langchain_tools]}")

Agent created with MCP-sourced tools.
  Tools: ['labuser17579966_1791942688__agent_training__classify_price_tier_']


### E3. Test the MCP-Powered Agent

With autologging enabled, each agent run produces a full execution trace. The trace captures the LLM reasoning step, the tool call to `classify_price_tier_uc` (now routed through MCP), and the final response generation.

In [0]:
traced_query = "I found a listing for $225/night in the Mission district. Is that a fair price?"
result = mcp_agent_executor.invoke(
    {
        "messages": [("human", traced_query)]
        }
    )
print(f"User:     {traced_query}")
print(f"Response: {result['messages'][-1].content}")

User:     I found a listing for $225/night in the Mission district. Is that a fair price?
Response: $225/night in SF classifies as **Premium** pricing, around the **50th–75th percentile** of the market.

- **Fair price?** Yes—it's **above average**, but still within a common range for SF, not “Luxury.”
- **What this implies:** You’re paying a bit extra versus a typical mid-range place; it’s reasonable if the Mission location, amenities, or quality are strong.


Trace(trace_id=trace:/labuser17579966_1791942688.agent_training.lc_sa/17ba60b517f637b3f34bf7d6e23abc52)

### E4. Run Additional Queries

In [0]:
result = mcp_agent_executor.invoke({"messages": [("human", query_1)]})
print(f"User:     {query_1}")
print(f"Response: {result['messages'][-1].content}")

User:     Is $150/night a good deal in SF?
Response: $150/night in San Francisco is **Mid-Range**, sitting in roughly the **25–50th percentile** of SF Airbnb prices—**around the market median**.

**Verdict:** It’s a **fair/typical deal**, not a standout bargain. If it includes strong location/amenities (or low fees), it can still be a good value.


Trace(trace_id=trace:/labuser17579966_1791942688.agent_training.lc_sa/b5b24645f662a065e11b217bd477ec59)

In [0]:
result = mcp_agent_executor.invoke({"messages": [("human", query_3)]})
print(f"User:     {query_3}")
print(f"Response: {result['messages'][-1].content}")

User:     Classify these prices: $80, $200, $350
Response: - **$80/night:** **Budget** (bottom **25%**) — well below SF market average.  
- **$200/night:** **Premium** (**50–75%**) — above average for SF listings.  
- **$350/night:** **Luxury** (top **25%**) — among the most expensive listings in SF.


Trace(trace_id=trace:/labuser17579966_1791942688.agent_training.lc_sa/343926a4f614b4cf2f758447ef14987e)

### E5. View MLflow Traces
Finally, now that we have some traces logged with MLflow, we can view the traces in either the **Experiments** UI or directly from the unified trace table stored in the schema `agent_training` with the table prefix `lc_sa`. 

#### Experiment UI
1. Navigate to **Experiments** in the left sidebar, right click, and select **Open Link in New Tab**. 
2. Search for the experiment **langchain-sdk-training**. There you will see traces under the **Traces** tab. Click any trace to see a detailed breakdown of tools used and model calls via **ChatDatabricks**.
#### MLflow Unified Tables from Unity Catalog
1. Navigate to the **Catalog Explorer** and search for **lc_sa**. This surfaces 6 default tables that have been created. 
    - 4 tables: **lc_sa_otel_logs**, **lc_sa_otel_spans**, **lc_sa_otel_annotations**, and **lc_sa_otel_annotations**
    - 2 views: **lc_sa_trace_unified**, and **lc_sa_trace_metadata**
2. Click on the **_otel_spans** table and click on **Sample Data** and ask a question in the provided query box like _Which trace took the longest to run?_ This will automatically generate a SQL query (you might be asked to grant auto-run, click accept). The answer will be returned to you.
3. Run the next cell to see logged traces here in the notebook. Notice the same information is captured in the column `spans`.

In [0]:
%sql
SELECT * FROM lc_sa_trace_unified

trace_id client_request_id request_time state execution_duration_ms request response trace_metadata tags spans assessments 0253521a79de2c12dffd2be6bb631d30 null 2026-10-14T03:46:00.753Z ERROR 5311.081667 {"messages":[["human","Is $150/night a good deal in SF?"]]} null {"mlflow.databricks.notebook.commandID":"1001791947306843_7683604188064789538_8779b87435524ca5a944e5c9106a2580","mlflow.databricks.notebookID":"3188534432723077","mlflow.databricks.notebookPath":"/Users/labuser17579966_1791942688@vocareum.com/building-agentic-applications-on-databricks-en_us-2.0.0/Building Agentic Applications on Databricks/07 Demo - Introduction to LangChain on Databricks","mlflow.databricks.webappURL":"https://dbc-358760f8-bd5d.cloud.databricks.com","mlflow.databricks.workspaceID":"7474660350819601","mlflow.databricks.workspaceURL":"https://dbc-358760f8-bd5d.cloud.databricks.com","mlflow.source.name":"/Users/labuser17579966_1791942688@vocareum.com/building-agentic-applications-on-databricks-en_us-2.0.0/Building Agentic Applications on Databricks/07 Demo - Introduction to LangChain on Databricks","mlflow.source.type":"NOTEBOOK","mlflow.trace.sizeBytes":"34661","mlflow.trace.sizeStats":"{\"total_size_bytes\": 34661, \"num_spans\": 7, \"max\": 8257, \"p25\": 1947, \"p50\": 4147, \"p75\": 7421}","mlflow.trace.tokenUsage":"{\"input_tokens\": 301, \"output_tokens\": 35, \"total_tokens\": 336, \"cache_read_input_tokens\": 0}","mlflow.traceInputs":"{\"messages\": [[\"human\", \"Is $150/night a good deal in SF?\"]]}","mlflow.traceOutputs":"","mlflow.trace_schema.version":"4","mlflow.user":"spark-8cbb9a5d-e6dc-41ac-9899-4e"} Map(mlflow.traceName -> LangGraph) List(List(01a13884-5bb4-719f-b786-e39f08841200, 2026-10-14T03:46:00.756Z, 2026-10-14, null, 0253521a79de2c12dffd2be6bb631d30, 8f5f2e91ba2d136a, null, 7575a272c43c3975, null, ChatDatabricks, null, 1791949560756137383, 1791949561804915929, {"batch_size":1,"invocation_params":{"_type":"chat-databricks","extra_params":{},"model":"labuser17579966_1791942688.agent_training.agent-training-model","n":1,"stop":null,"tools":[{"function":{"description":"Classify an Airbnb listing price into a market tier based on SF pricing distribution. Uses hardcoded SF Airbnb market thresholds to label a nightly price as Budget, Mid-Range, Premium, or Luxury, and returns the corresponding percentile band for context. Useful for questions like: \"Is $180/night a good deal in SF?\", \"What tier does this listing fall into?\" Returns a JSON object with tier, percentile_band, and a short interpretation.","name":"labuser17579966_1791942688__agent_training__classify_price_tier_uc","parameters":{"properties":{"price":{"description":"Nightly listing price in USD.","type":"number"}},"required":["price"],"type":"object"}},"type":"function"}]},"metadata":{"checkpoint_ns":"model:ca4b0343-d019-05fb-79f7-61ac550e7df9","langgraph_checkpoint_ns":"model:ca4b0343-d019-05fb-79f7-61ac550e7df9","langgraph_node":"model","langgraph_path":["__pregel_pull","model"],"langgraph_step":1,"langgraph_triggers":["branch:to:model"],"lc_versions":{"langchain":"1.4.3","langchain-core":"1.6.6"},"ls_integration":"langchain_chat_model","ls_model_name":"labuser17579966_1791942688.agent_training.agent-training-model","ls_model_type":"chat","ls_provider":"databricks"},"mlflow.chat.tokenUsage":{"cache_read_input_tokens":0,"input_tokens":301,"output_tokens":35,"total_tokens":336},"mlflow.chat.tools":[{"function":{"description":"Classify an Airbnb listing price into a market tier based on SF pricing distribution. Uses hardcoded SF Airbnb market thresholds to label a nightly price as Budget, Mid-Range, Premium, or Luxury, and returns the corresponding percentile band for context. Useful for questions like: \"Is $180/night a good deal in SF?\", \"What tier does this listing fall into?\" Returns a JSON object with tier, percentile_band, and a short interpretation.","name":"labuser17579966_1791942688__agent_training__classify_price_tier_uc","parameters":{"properties":{"p

## F. Conclusion

In this demo, you:

1. Defined a `classify_price_tier` tool using LangChain's `@tool` decorator to classify Airbnb prices into market tiers
2. Built a single Data Analyst agent with LangChain's `create_agent` and tested it with progressively complex pricing queries
3. Registered `classify_price_tier` as a Python UC function for governance
4. Discovered the UC function through a Managed MCP server and wrapped it as a LangChain `StructuredTool`
5. Rebuilt the agent with MCP-sourced tools and enabled MLflow tracing with `mlflow.langchain.autolog()`

The same `@tool` to UC to MCP progression works across frameworks -- whether you use LangChain (this demo) or the OpenAI Agents SDK (notebook 05).

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>